In [1]:
from dataset import Dataset
from model import Retriever, Augmenter, Generator, RetrievalEvaluator
from evaluate import Evaluator
import os

import warnings
warnings.filterwarnings("ignore")
import logging
logging.getLogger("transformers.modeling_utils").setLevel(logging.ERROR)
import absl.logging
absl.logging.set_verbosity(absl.logging.ERROR)

In [2]:
note_prompts = {
    "easy": "Important Note: Your output will strictly be Yes or No with no other words.",
    "medium": "Important Note: Your output must be strictly, with no extra words, separated by comma, a list of nutrients with high or low before the nutrients among these options: carb, protein, sugar, sodium, cholesterol, \
        saturated_fat, calorie. For example, the output is: high_carb, low_protein, high_sugar.\
        You should only include the nutrient tags that connect the food with the user.",
    "hard": "Important Note: Your output must be a Yes or No followed by strictly a list of nutrients with high or low as prefix among these options: carb, protein, sugar, sodium, cholesterol, \
        saturated fat, calorie. For example, the output is: Yes, because the food is high in carb, low in protein, high in sugar.",
}

method_prompts = {
    "plain": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "KAPPING": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "ToG": "Below are the extra information you use to answer the question, note that you should not use your general knowledge and the answer is among this information.",
    "Zero_CoT": "Let's think step by step",
    "CoT_BaG": ""
}

## ToG

In [ ]:

# Modify the parameters here to find good prompts.
api_key = os.getenv("OPENAI_API_KEY")
# api_key = os.getenv('LLAMA_API_KEY')
file_path = "../processed_data/NutriGraphQA_benchmark.csv"

task_level = "medium"
question_level = "easy"
is_sample = True
n = 10
# model_name = "llama3.1-70b"
model_name = "gpt-4o-mini"
method = "ToG"

note_prompt = note_prompts.get(task_level)
method_prompt = method_prompts.get(method)

data = Dataset(file_path)
questions, answers, graphs = data.process(question_level=question_level, task_level=task_level, sample=is_sample, n=n)

retriever = Retriever(graphs, model_name=model_name)
retrieved_graphs = retriever.retrieve(method=method, api_key=api_key, questions=questions)

retrieval_evaluator = RetrievalEvaluator(graphs)
retrieval_evaluation_results = retrieval_evaluator.evaluate(retrieved_graphs)
print('Retrieval evaluation results:', retrieval_evaluation_results)

augmenter = Augmenter()
textualized_graphs = augmenter.augment(retrieved_graphs)

generator = Generator(api_key = api_key, 
                      model_name=model_name, note_prompt=note_prompt, method_prompt=method_prompt)
predictions = generator.generate_predictions(questions, textualized_graphs)

evaluator = Evaluator()
final_output_results = evaluator.evaluate(task_level, predictions, answers)
print('Final output evaluation results:', final_output_results)

## CoT BaG

In [3]:
from utils import generate_paragraph_cot_bag

api_key = os.getenv("OPENAI_API_KEY")
file_path = "../processed_data/NutriGraphQA_benchmark.csv"

task_level = "easy"
question_level = "medium"
is_sample = True
n = 50
model_name = "gpt-4o-mini"
method = "CoT_BaG"

note_prompt = note_prompts.get(task_level)
method_prompt = method_prompts.get(method)

data = Dataset(file_path)
questions, answers, graphs = data.process(question_level=question_level, task_level=task_level, sample=is_sample, n=n)

retriever = Retriever(graphs, model_name=model_name)
retrieved_graphs = retriever.retrieve(method=method, api_key=api_key, questions=questions)

retrieval_evaluator = RetrievalEvaluator(graphs)
retrieval_evaluation_results = retrieval_evaluator.evaluate(retrieved_graphs)
print('Retrieval evaluation results:', retrieval_evaluation_results)

augmenter = Augmenter()
textualized_graphs = augmenter.augment(retrieved_graphs)

# Convert to paragraph format for CoT_BaG
if method == "CoT_BaG":
    textualized_graphs = [generate_paragraph_cot_bag(graph) for graph in textualized_graphs]

generator = Generator(api_key=api_key, model_name=model_name, note_prompt=note_prompt, method_prompt=method_prompt)
predictions = generator.generate_predictions(questions, textualized_graphs)

evaluator = Evaluator()
final_output_results = evaluator.evaluate(task_level, predictions, answers)
print('Final output evaluation results:', final_output_results)


Retrieving Subgraphs: 100%|██████████| 50/50 [00:00<00:00, 454913.67it/s]


Retrieval evaluation results: {'Precision': 0.238, 'Recall': 1.0, 'F1 Score': 0.382}


Generating Predictions:   0%|          | 0/50 [00:00<?, ?it/s]

Prompt:  Based on the nutrients the food provides and the user needs, please answer if the food 28315140 is a healthy option to the user 77843? Please answer with yes or no.. . You are given a directed graph where the nodes and edges are: an edge between "Beef vegetable soup, home recipe, Mexican style" directed to "Soups" with attribute "belongs to", an edge between "Beef vegetable soup, home recipe, Mexican style" directed to "Beef, chuck, arm pot roast, separable lean only, trimmed to 1/8" fat, choice, cooked, braised" with attribute "has", an edge between "Beef vegetable soup, home recipe, Mexican style" directed to "Vegetable oil, NFS" with attribute "has", an edge between "Beef vegetable soup, home recipe, Mexican style" directed to "Potatoes, flesh and skin, raw" with attribute "has", an edge between "Beef vegetable soup, home recipe, Mexican style" directed to "Corn, sweet, yellow, raw" with attribute "has", an edge between "Beef vegetable soup, home recipe, Mexican style" dire

Generating Predictions:   2%|▏         | 1/50 [00:04<03:27,  4.23s/it]

...

Answer: No
Prompt:  Based on the nutrients the food provides and the user needs, please answer if the food 41311020 is a healthy option to the user 62063? Please answer with yes or no.. . You are given a directed graph where the nodes and edges are: an edge between "Sambar, vegetable stew" directed to "Vegetable dishes" with attribute "belongs to", an edge between "Sambar, vegetable stew" directed to "Spices, chili powder" with attribute "has", an edge between "Sambar, vegetable stew" directed to "Salt, table" with attribute "has", an edge between "Sambar, vegetable stew" directed to "Water, tap, drinking" with attribute "has", an edge between "Sambar, vegetable stew" directed to "Spices, chili powder" with attribute "has", an edge between "Sambar, vegetable stew" directed to "Spices, turmeric, ground" with attribute "has", an edge between "Sambar, vegetable stew" directed to "Lentils, mature seeds, cooked, boiled, without salt" with attribute "has", an edge between "Sambar, vegetable 

Generating Predictions:  98%|█████████▊| 49/50 [01:26<00:01,  1.61s/it]

Answer: No
Prompt:  Based on the nutrients the food provides and the user needs, please answer if the food 27460750 is a healthy option to the user 96547? Please answer with yes or no.. . You are given a directed graph where the nodes and edges are: an edge between "Liver, beef or calves, and onions" directed to "Meat mixed dishes" with attribute "belongs to", an edge between "Liver, beef or calves, and onions" directed to "Beef, variety meats and by-products, liver, cooked, pan-fried" with attribute "has", an edge between "Liver, beef or calves, and onions" directed to "Onions, cooked, boiled, drained, without salt" with attribute "has", an edge between "Liver, beef or calves, and onions" directed to "Salt, table, iodized" with attribute "has", an edge between "Liver, beef or calves, and onions" directed to "Margarine, stick" with attribute "has", an edge between "Liver, beef or calves, and onions" directed to "low_carb" with attribute "belongs to", an edge between "Liver, beef or cal

Generating Predictions: 100%|██████████| 50/50 [01:28<00:00,  1.78s/it]

Answer: Yes
Final output evaluation results: {'Accuracy': 0.46, 'Precision': 1.0, 'Recall': 0.1818, 'F1 Score': 0.3077, 'AUC Score': 0.5909}


## CoT

In [4]:
api_key = os.getenv("OPENAI_API_KEY")
file_path = "../processed_data/NutriGraphQA_benchmark.csv"

task_level = "easy"
question_level = "medium"
is_sample = True
n = 50
model_name = "gpt-4o-mini"
method = "Zero_CoT"

note_prompt = note_prompts.get(task_level)
method_prompt = method_prompts.get(method)

data = Dataset(file_path)
questions, answers, graphs = data.process(question_level=question_level, task_level=task_level, sample=is_sample, n=n)

retriever = Retriever(graphs, model_name=model_name)
retrieved_graphs = retriever.retrieve(method=method, api_key=api_key, questions=questions)

retrieval_evaluator = RetrievalEvaluator(graphs)
retrieval_evaluation_results = retrieval_evaluator.evaluate(retrieved_graphs)
print('Retrieval evaluation results:', retrieval_evaluation_results)

augmenter = Augmenter()
textualized_graphs = augmenter.augment(retrieved_graphs)

generator = Generator(api_key=api_key, model_name=model_name, note_prompt=note_prompt, method_prompt=method_prompt)
predictions = generator.generate_predictions(questions, textualized_graphs)

evaluator = Evaluator()
final_output_results = evaluator.evaluate(task_level, predictions, answers)
print('Final output evaluation results:', final_output_results)

Retrieving Subgraphs: 100%|██████████| 50/50 [00:00<00:00, 1133595.68it/s]


Retrieval evaluation results: {'Precision': 0.238, 'Recall': 1.0, 'F1 Score': 0.382}


Generating Predictions:   0%|          | 0/50 [00:00<?, ?it/s]

Prompt:  Based on the nutrients the food provides and the user needs, please answer if the food 28315140 is a healthy option to the user 77843? Please answer with yes or no.. Let's think step by step. (Beef vegetable soup, home recipe, Mexican style belongs to Soups), (Beef vegetable soup, home recipe, Mexican style has Beef, chuck, arm pot roast, separable lean only, trimmed to 1/8" fat, choice, cooked, braised), (Beef vegetable soup, home recipe, Mexican style has Vegetable oil, NFS), (Beef vegetable soup, home recipe, Mexican style has Potatoes, flesh and skin, raw), (Beef vegetable soup, home recipe, Mexican style has Corn, sweet, yellow, raw), (Beef vegetable soup, home recipe, Mexican style has Tomatoes, red, ripe, canned, packed in tomato juice), (Beef vegetable soup, home recipe, Mexican style has Carrots, raw), (Beef vegetable soup, home recipe, Mexican style has Cabbage, raw), (Beef vegetable soup, home recipe, Mexican style has Salt, table, iodized), (Beef vegetable soup, ho

Generating Predictions:   2%|▏         | 1/50 [00:05<04:07,  5.05s/it]

...

Answer: No
Prompt:  Based on the nutrients the food provides and the user needs, please answer if the food 41311020 is a healthy option to the user 62063? Please answer with yes or no.. Let's think step by step. (Sambar, vegetable stew belongs to Vegetable dishes), (Sambar, vegetable stew has Spices, chili powder), (Sambar, vegetable stew has Salt, table), (Sambar, vegetable stew has Water, tap, drinking), (Sambar, vegetable stew has Spices, chili powder), (Sambar, vegetable stew has Spices, turmeric, ground), (Sambar, vegetable stew has Lentils, mature seeds, cooked, boiled, without salt), (Sambar, vegetable stew has Vegetable oil, NFS), (Sambar, vegetable stew has Okra, frozen, cooked, boiled, drained, without salt), (Sambar, vegetable stew has Onions, cooked, as ingredient), (Sambar, vegetable stew has Potatoes, baked, flesh and skin, without salt), (Sambar, vegetable stew has Tomatoes, cooked, as ingredient), (Sambar, vegetable stew has Tamarind, dried), (Sambar, vegetable stew has

Generating Predictions:  98%|█████████▊| 49/50 [01:26<00:01,  1.76s/it]

Answer: No
Prompt:  Based on the nutrients the food provides and the user needs, please answer if the food 27460750 is a healthy option to the user 96547? Please answer with yes or no.. Let's think step by step. (Liver, beef or calves, and onions belongs to Meat mixed dishes), (Liver, beef or calves, and onions has Beef, variety meats and by-products, liver, cooked, pan-fried), (Liver, beef or calves, and onions has Onions, cooked, boiled, drained, without salt), (Liver, beef or calves, and onions has Salt, table, iodized), (Liver, beef or calves, and onions has Margarine, stick), (Liver, beef or calves, and onions belongs to low_carb), (Liver, beef or calves, and onions belongs to low_sugar), (Liver, beef or calves, and onions belongs to high_sodium), (Liver, beef or calves, and onions belongs to high_protein), (Liver, beef or calves, and onions belongs to high_cholesterol), (Liver, beef or calves, and onions belongs to low_saturated_fat), (user has Eats little or no fish), (user has 

Generating Predictions: 100%|██████████| 50/50 [01:28<00:00,  1.76s/it]

Answer: No
Final output evaluation results: {'Accuracy': 0.46, 'Precision': 1.0, 'Recall': 0.1818, 'F1 Score': 0.3077, 'AUC Score': 0.5909}
